# Notebook 2 - Train a Probe on One Dataset

This notebook runs the first complete Phi-2 probing experiment. It uses one dataset, extracts Phi-2 hidden states, trains a probe, and evaluates train/validation/test grouped accuracy.

This is an in-distribution sanity check. It tells us whether the probes can learn a useful signal on held-out examples from the same dataset before we ask whether they transfer to other datasets.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "lie_detector_llm").exists():
            return candidate
    raise RuntimeError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RESULTS_DIR = PROJECT_ROOT / "results"
ACTIVATION_CACHE_DIR = PROJECT_ROOT / "data" / "activations"
RESULTS_DIR.mkdir(exist_ok=True)
ACTIVATION_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")


## Dataset Choice

We start with `dbpedia_14`. This is a topic-classification dataset reformatted as candidate-answer correctness. It is a useful source dataset because it gives clean contrast groups: one correct category and several incorrect categories.

In this project, `dbpedia_14` is also used as the default source dataset for transfer experiments because it produces strong transfer in our Phi-2 run.


In [ ]:
from lie_detector_llm.datasets import build_dataset_collection

DATASET_NAME = "dbpedia_14"
MAX_GROUPS = 80

collection = build_dataset_collection(
    dataset_names=[DATASET_NAME],
    max_groups=MAX_GROUPS,
    seed=0,
)
frame = collection.subset(DATASET_NAME)

display(collection.summary())
print("Rows:", len(frame))
print("Groups:", frame["group_id"].nunique())


## Model and Layer

The model is fixed to `microsoft/phi-2`. The layer below is set to `18` because the layer sweep in Notebook 4 found that layer 18 gives the best out-of-distribution performance for `DIM` and `PCA-G`.

The activations are cached in `data/activations/`, so rerunning this notebook should be fast after the first extraction.


In [ ]:
from lie_detector_llm.experiment import DEFAULT_MODEL

MODEL_NAME = DEFAULT_MODEL
PROBE_METHOD = "dim"
LAYER_INDEX = 18
ACTIVATION_BATCH_SIZE = 2
MAX_LENGTH = 512
LOAD_IN_4BIT = False

print("Model:", MODEL_NAME)
print("Probe:", PROBE_METHOD)
print("Layer:", LAYER_INDEX)


## Single-Probe Run

`run_probe_experiment` performs the whole pipeline for one probe: group split, activation loading/extraction, probe training, scoring, and grouped accuracy.

The test split is the relevant same-dataset generalisation score.


In [ ]:
from lie_detector_llm.experiment import run_probe_experiment

single = run_probe_experiment(
    frame=frame,
    model_name=MODEL_NAME,
    probe_method=PROBE_METHOD,
    layer_index=LAYER_INDEX,
    activation_batch_size=ACTIVATION_BATCH_SIZE,
    max_length=MAX_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    show_progress=True,
    activation_cache_dir=ACTIVATION_CACHE_DIR,
)

display(single.results)


## Compare the Four Probes

The report uses four probes: `dim`, `lat`, `lr`, and `pca-g`. This cell compares them on the same train/validation/test split.

A high test score here does not prove transfer. It only shows that the probe can generalise within the same dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from lie_detector_llm.experiment import PROBE_METHODS, run_probe_comparison

comparison = run_probe_comparison(
    frame=frame,
    probe_methods=PROBE_METHODS,
    model_name=MODEL_NAME,
    layer_index=LAYER_INDEX,
    activation_batch_size=ACTIVATION_BATCH_SIZE,
    max_length=MAX_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    activation_cache_dir=ACTIVATION_CACHE_DIR,
)

display(comparison.results)

test_rows = comparison.results[comparison.results["split"] == "test"].copy()
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=test_rows, x="probe_method", y="grouped_accuracy", ax=ax)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Probe")
ax.set_ylabel("Grouped accuracy")
ax.set_title(f"Phi-2 probe comparison on {DATASET_NAME}, layer {LAYER_INDEX}")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "phi2_single_dataset_probe_comparison_plot.png", dpi=160, bbox_inches="tight")
plt.show()


## Observation Guide

Use this notebook to check basic probe behaviour. If train accuracy is high but validation/test are much lower, the probe is probably overfitting. If train, validation, and test are all high, the dataset contains a readable linear signal at this layer.

In the current Phi-2 run, `dbpedia_14` at layer 18 is a strong same-dataset setup for `DIM`, `LR`, and `PCA-G`. Transfer is tested in the next notebooks.
